# Problem: Attention Visualization in Transformers

### Problem Statement
You are tasked with implementing a **Transformer model** from scratch and training it on a synthetic sequence-to-sequence task. After training, you must **visualize the attention patterns** learned by different attention heads to understand how the model processes input sequences.

### Requirements
1. **Data Acquisition**: Generate synthetic sequence data for a simple task (e.g., sequence copying, reversal, or sorted sequence prediction).
2. **Data Wrangling**: Prepare the synthetic data with proper tokenization and padding.
3. **Model Implementation**: Implement a Transformer model with multi-head attention from scratch using PyTorch.
4. **Training**: Train the model on the synthetic task using appropriate loss function and optimizer.
5. **Evaluation**: Evaluate model performance and visualize attention maps for different heads.

### Constraints
- Implement the Transformer architecture including multi-head attention mechanism.
- Generate at least 1000 training examples.
- Use at least 2 attention heads.
- Visualize attention patterns using heatmaps.

### Learning Objectives
- Understand the internal mechanics of Transformer attention.
- Learn how different attention heads specialize in different patterns.
- Practice implementing complex neural architectures from scratch.

<details>
  <summary>💡 Hint</summary>
  Start with a simple sequence task like copying or reversing sequences. Implement scaled dot-product attention first, then build multi-head attention on top. Use matplotlib or seaborn to create attention heatmaps. Extract attention weights during the forward pass for visualization.
</details>

## Step 1: Data Acquisition

Generate synthetic sequence data for training the Transformer model.

**Task**: Create a dataset where the model learns to reverse sequences of integers.
- Input: Random sequences of integers (e.g., [1, 5, 3, 7, 2])
- Output: Reversed sequences (e.g., [2, 7, 3, 5, 1])
- Vocabulary size: 10-20 tokens
- Sequence length: 5-10 tokens
- Dataset size: 1000+ examples

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Dataset, DataLoader
import math

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Generate synthetic sequence reversal dataset
def generate_reversal_dataset(num_samples=2000, seq_length=8, vocab_size=15):
    """
    Generate sequences and their reversals.
    vocab_size includes regular tokens (3 to vocab_size-1)
    """
    sequences = []
    targets = []
    
    for _ in range(num_samples):
        # Generate random sequence (starting from token 3 to avoid special tokens)
        seq = np.random.randint(3, vocab_size, size=seq_length)
        # Reverse it
        rev_seq = seq[::-1].copy()
        
        sequences.append(seq)
        targets.append(rev_seq)
    
    return np.array(sequences), np.array(targets)

# Generate dataset
VOCAB_SIZE = 15
SEQ_LENGTH = 8
NUM_SAMPLES = 2000

sequences, targets = generate_reversal_dataset(NUM_SAMPLES, SEQ_LENGTH, VOCAB_SIZE)

print(f"Generated {len(sequences)} sequence pairs")
print(f"Example input: {sequences[0]}")
print(f"Example target: {targets[0]}")
print(f"Vocabulary size: {VOCAB_SIZE}")

## Step 2: Data Wrangling

Prepare the data for training:
- Add special tokens (PAD, SOS, EOS)
- Create PyTorch Dataset and DataLoader
- Implement proper batching and padding

In [ ]:
# Define special tokens
PAD_TOKEN = 0
SOS_TOKEN = 1  # Start of sequence
EOS_TOKEN = 2  # End of sequence

print(f"Special tokens: PAD={PAD_TOKEN}, SOS={SOS_TOKEN}, EOS={EOS_TOKEN}")

In [ ]:
# Create PyTorch Dataset class
class SequenceDataset(Dataset):
    def __init__(self, sequences, targets):
        self.sequences = sequences
        self.targets = targets
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        # Source: add EOS at the end
        src = np.concatenate([self.sequences[idx], [EOS_TOKEN]])
        
        # Target: add SOS at the beginning and EOS at the end
        tgt_input = np.concatenate([[SOS_TOKEN], self.targets[idx]])
        tgt_output = np.concatenate([self.targets[idx], [EOS_TOKEN]])
        
        return (
            torch.LongTensor(src),
            torch.LongTensor(tgt_input),
            torch.LongTensor(tgt_output)
        )

In [ ]:
# Split into train and validation sets
split_idx = int(0.8 * len(sequences))
train_sequences, val_sequences = sequences[:split_idx], sequences[split_idx:]
train_targets, val_targets = targets[:split_idx], targets[split_idx:]

# Create datasets
train_dataset = SequenceDataset(train_sequences, train_targets)
val_dataset = SequenceDataset(val_sequences, val_targets)

# Create dataloaders
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Batch size: {BATCH_SIZE}")

## Step 3: Model Implementation

Implement a Transformer model with the following components:
1. **Positional Encoding**: Add position information to embeddings
2. **Multi-Head Attention**: Implement scaled dot-product attention with multiple heads
3. **Encoder**: Stack of attention and feedforward layers
4. **Decoder**: Stack of masked attention, encoder-decoder attention, and feedforward layers
5. **Output Layer**: Linear projection to vocabulary

In [ ]:
# Implement Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super(PositionalEncoding, self).__init__()
        
        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        pe = pe.unsqueeze(0)  # Add batch dimension
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        # x shape: (batch_size, seq_len, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return x

In [ ]:
# Implement Multi-Head Attention
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Linear layers for Q, K, V projections
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        # Q, K, V shape: (batch_size, num_heads, seq_len, d_k)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attention_weights = torch.softmax(scores, dim=-1)
        output = torch.matmul(attention_weights, V)
        
        return output, attention_weights
    
    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)
        
        # Linear projections and reshape to (batch_size, num_heads, seq_len, d_k)
        Q = self.W_q(query).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(key).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(value).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        # Apply attention
        attn_output, attention_weights = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # Concatenate heads
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        
        # Final linear projection
        output = self.W_o(attn_output)
        
        return output, attention_weights

In [ ]:
# Implement Feedforward Network
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super(FeedForward, self).__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        return self.linear2(self.dropout(self.relu(self.linear1(x))))

In [ ]:
# Implement Transformer Encoder Layer
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super(TransformerEncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Self-attention with residual connection
        attn_output, _ = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        
        # Feedforward with residual connection
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        
        return x

In [ ]:
# Implement Transformer Decoder Layer
class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super(TransformerDecoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        # Masked self-attention
        attn_output, self_attn_weights = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(attn_output))
        
        # Cross-attention with encoder output
        attn_output, cross_attn_weights = self.cross_attn(x, enc_output, enc_output, src_mask)
        x = self.norm2(x + self.dropout(attn_output))
        
        # Feedforward
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_output))
        
        return x, self_attn_weights, cross_attn_weights

In [ ]:
# Implement complete Transformer model
class Transformer(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, d_ff, max_len=100, dropout=0.1):
        super(Transformer, self).__init__()
        
        self.d_model = d_model
        self.num_heads = num_heads
        
        # Embeddings
        self.src_embedding = nn.Embedding(vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        
        # Encoder and Decoder layers
        self.encoder_layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        
        self.decoder_layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        
        # Output projection
        self.fc_out = nn.Linear(d_model, vocab_size)
        self.dropout = nn.Dropout(dropout)
        
    def generate_mask(self, src, tgt):
        # Source mask (for padding)
        src_mask = (src != PAD_TOKEN).unsqueeze(1).unsqueeze(2)
        
        # Target mask (for padding and future positions)
        tgt_len = tgt.size(1)
        tgt_mask = torch.tril(torch.ones((tgt_len, tgt_len), device=tgt.device)).bool()
        tgt_mask = tgt_mask.unsqueeze(0).unsqueeze(1)
        tgt_padding_mask = (tgt != PAD_TOKEN).unsqueeze(1).unsqueeze(2)
        tgt_mask = tgt_mask & tgt_padding_mask
        
        return src_mask, tgt_mask
    
    def encode(self, src, src_mask):
        x = self.dropout(self.pos_encoding(self.src_embedding(src) * math.sqrt(self.d_model)))
        
        for layer in self.encoder_layers:
            x = layer(x, src_mask)
        
        return x
    
    def decode(self, tgt, enc_output, src_mask, tgt_mask):
        x = self.dropout(self.pos_encoding(self.tgt_embedding(tgt) * math.sqrt(self.d_model)))
        
        self_attn_weights_list = []
        cross_attn_weights_list = []
        
        for layer in self.decoder_layers:
            x, self_attn_weights, cross_attn_weights = layer(x, enc_output, src_mask, tgt_mask)
            self_attn_weights_list.append(self_attn_weights)
            cross_attn_weights_list.append(cross_attn_weights)
        
        return x, self_attn_weights_list, cross_attn_weights_list
    
    def forward(self, src, tgt, return_attention=False):
        src_mask, tgt_mask = self.generate_mask(src, tgt)
        
        # Encode
        enc_output = self.encode(src, src_mask)
        
        # Decode
        dec_output, self_attn_weights_list, cross_attn_weights_list = self.decode(
            tgt, enc_output, src_mask, tgt_mask
        )
        
        # Output projection
        output = self.fc_out(dec_output)
        
        if return_attention:
            return output, self_attn_weights_list, cross_attn_weights_list
        return output

## Step 4: Training

Train the Transformer model:
- Loss function: Cross-entropy loss
- Optimizer: Adam with learning rate scheduling
- Track training loss and accuracy
- Implement teacher forcing for decoder training

In [ ]:
# Initialize model, loss function, and optimizer
D_MODEL = 128
NUM_HEADS = 4
NUM_LAYERS = 2
D_FF = 512
DROPOUT = 0.1

model = Transformer(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    d_ff=D_FF,
    dropout=DROPOUT
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_TOKEN)
optimizer = optim.Adam(model.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9)

print(f"Model parameters: {sum(p.numel() for p in model.parameters())}")
print(f"Model architecture:")
print(f"  - d_model: {D_MODEL}")
print(f"  - num_heads: {NUM_HEADS}")
print(f"  - num_layers: {NUM_LAYERS}")
print(f"  - d_ff: {D_FF}")

In [ ]:
# Implement training loop with teacher forcing
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for src, tgt_input, tgt_output in dataloader:
        src = src.to(device)
        tgt_input = tgt_input.to(device)
        tgt_output = tgt_output.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        output = model(src, tgt_input)
        
        # Calculate loss
        loss = criterion(output.view(-1, VOCAB_SIZE), tgt_output.view(-1))
        
        # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
        
        # Calculate accuracy
        pred = output.argmax(dim=-1)
        mask = (tgt_output != PAD_TOKEN)
        correct += ((pred == tgt_output) & mask).sum().item()
        total += mask.sum().item()
    
    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for src, tgt_input, tgt_output in dataloader:
            src = src.to(device)
            tgt_input = tgt_input.to(device)
            tgt_output = tgt_output.to(device)
            
            output = model(src, tgt_input)
            loss = criterion(output.view(-1, VOCAB_SIZE), tgt_output.view(-1))
            
            total_loss += loss.item()
            
            pred = output.argmax(dim=-1)
            mask = (tgt_output != PAD_TOKEN)
            correct += ((pred == tgt_output) & mask).sum().item()
            total += mask.sum().item()
    
    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

In [ ]:
# Train the model for multiple epochs
NUM_EPOCHS = 30

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

print("Starting training...")
print()

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{NUM_EPOCHS}]")
        print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        print(f"  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
        print()

print("Training completed!")

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_losses, label='Train Loss')
ax1.plot(val_losses, label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True)

ax2.plot(train_accuracies, label='Train Accuracy')
ax2.plot(val_accuracies, label='Val Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training and Validation Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## Step 5: Evaluation and Attention Visualization

Evaluate the model and visualize attention patterns:
1. Test model accuracy on validation set
2. Extract attention weights from different heads
3. Create heatmaps showing attention patterns
4. Analyze how different heads focus on different parts of the input

In [ ]:
# Evaluate model accuracy
final_val_loss, final_val_acc = evaluate(model, val_loader, criterion, device)
print(f"Final Validation Accuracy: {final_val_acc:.2f}%")
print(f"Final Validation Loss: {final_val_loss:.4f}")

In [ ]:
# Extract attention weights for a sample input
def get_attention_weights(model, src, tgt, device):
    model.eval()
    with torch.no_grad():
        src = src.unsqueeze(0).to(device)
        tgt = tgt.unsqueeze(0).to(device)
        
        output, self_attn_weights, cross_attn_weights = model(src, tgt, return_attention=True)
        
    return output, self_attn_weights, cross_attn_weights

# Get a sample from validation set
sample_idx = 5
src_sample, tgt_input_sample, tgt_output_sample = val_dataset[sample_idx]

print("Sample input sequence:", src_sample.numpy())
print("Sample target sequence:", tgt_output_sample.numpy())

# Get attention weights
output, self_attn_weights, cross_attn_weights = get_attention_weights(
    model, src_sample, tgt_input_sample, device
)

print(f"\nNumber of decoder layers: {len(cross_attn_weights)}")
print(f"Number of attention heads: {NUM_HEADS}")

In [ ]:
# Visualize attention maps for different heads
def plot_attention_maps(attention_weights, src_tokens, tgt_tokens, layer_idx=0):
    """
    Plot attention maps for all heads in a specific layer.
    attention_weights: (num_heads, tgt_len, src_len)
    """
    num_heads = attention_weights.size(0)
    
    fig, axes = plt.subplots(1, num_heads, figsize=(5 * num_heads, 4))
    if num_heads == 1:
        axes = [axes]
    
    for head_idx in range(num_heads):
        ax = axes[head_idx]
        attn_map = attention_weights[head_idx].cpu().numpy()
        
        sns.heatmap(attn_map, cmap='viridis', ax=ax, cbar=True,
                    xticklabels=src_tokens, yticklabels=tgt_tokens,
                    vmin=0, vmax=1)
        
        ax.set_title(f'Layer {layer_idx} - Head {head_idx}')
        ax.set_xlabel('Source Position')
        ax.set_ylabel('Target Position')
    
    plt.tight_layout()
    plt.show()

# Convert tokens to strings for visualization
def tokens_to_str(tokens):
    return [f"{t}" if t >= 3 else ['PAD', 'SOS', 'EOS'][t] for t in tokens]

src_str = tokens_to_str(src_sample.numpy())
tgt_str = tokens_to_str(tgt_output_sample.numpy())

In [ ]:
# Visualize cross-attention for each layer
for layer_idx in range(len(cross_attn_weights)):
    attn = cross_attn_weights[layer_idx].squeeze(0)  # Remove batch dimension
    print(f"\n=== Decoder Layer {layer_idx} Cross-Attention ===")
    plot_attention_maps(attn, src_str, tgt_str, layer_idx)

In [ ]:
# Test on multiple examples to see different attention patterns
def visualize_multiple_examples(model, dataset, num_examples=3):
    for idx in range(num_examples):
        print(f"\n{'='*80}")
        print(f"Example {idx + 1}")
        print(f"{'='*80}")
        
        src_sample, tgt_input_sample, tgt_output_sample = dataset[idx]
        
        print("Input:", src_sample.numpy())
        print("Target:", tgt_output_sample.numpy())
        
        # Get predictions
        output, _, cross_attn_weights = get_attention_weights(
            model, src_sample, tgt_input_sample, device
        )
        pred = output.squeeze(0).argmax(dim=-1).cpu().numpy()
        print("Predicted:", pred)
        
        # Visualize last layer cross-attention
        attn = cross_attn_weights[-1].squeeze(0)
        src_str = tokens_to_str(src_sample.numpy())
        tgt_str = tokens_to_str(tgt_output_sample.numpy())
        
        plot_attention_maps(attn, src_str, tgt_str, len(cross_attn_weights) - 1)

# Visualize attention for multiple examples
visualize_multiple_examples(model, val_dataset, num_examples=3)

In [ ]:
# Analyze attention patterns across heads
print("\nAttention Pattern Analysis:")
print("=" * 50)
print(f"Model with {NUM_HEADS} attention heads trained to reverse sequences.")
print(f"\nKey Observations:")
print(f"1. Different heads learn different positional correspondences")
print(f"2. Some heads focus on specific relative positions")
print(f"3. Attention patterns show the reversal mapping learned by the model")
print(f"4. Darker colors indicate stronger attention weights")
print(f"\nFor sequence reversal:")
print(f"- First target position should attend to last source position")
print(f"- Second target position should attend to second-to-last source position")
print(f"- And so on...")